In [1]:
!pip install pydantic-ai[google] pymupdf pgvector psycopg[binary]


# ⚡ Pydantic AI + Gemini PDF RAG (Ponytail Edition)
불필요한 벡터DB, 임베딩 파이프라인 없이 **20줄**로 끝내는 가장 간결한 RAG

In [2]:
import os
from pathlib import Path
import psycopg
from pgvector.psycopg import register_vector
import pymupdf
from pydantic_ai.embeddings.google import GoogleEmbeddingModel
from pydantic_ai.providers.google import GoogleProvider
from pydantic_ai import Embedder
import re
from pydantic_ai.exceptions import ModelHTTPError

In [3]:

def Connection():
    connection = psycopg.connect(
        host="127.0.0.1",
        port=5432,
        user="postgres",
        password="1234",
        dbname="postgres"
    )
    register_vector(connection)#벡터를 float list로 변경해줌
    return connection
    


# 임베딩 모델 만들기

In [4]:
provider=GoogleProvider(api_key=os.getenv('gemini'))
model=GoogleEmbeddingModel("gemini-embedding-001",provider=provider)
embedder = Embedder(model)
embedder

Embedder(instrument=None)

# 차원 가져오기

In [5]:
sample = await embedder.embed_documents(".")
dim=len(sample.embeddings[0])
dim

3072

# Vector DB설정

In [7]:
with Connection() as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS pdf_documents (
                id SERIAL PRIMARY KEY,
                content TEXT,
                embedding vector({dim})
            );
        """)
    conn.commit()

# pdf 수집

In [8]:
pdf_files = list(Path("./pdf").glob("*.pdf"))
pdf_files

[PosixPath('pdf/2025홀수.pdf'),
 PosixPath('pdf/2026홀수.pdf'),
 PosixPath('pdf/2025짝수.pdf'),
 PosixPath('pdf/2026짝수.pdf')]

In [ ]:
all_chunks = []
for pdf_file in pdf_files:
    doc = pymupdf.open(pdf_file)
    for page in doc:
        text = page.get_text()
        
        # 1. '숫자. ' 패턴을 기준으로 자르기
        raw_chunks = re.split(r'\n(?=\d+\.\s)', text)
        
        for chunk in raw_chunks:
            chunk = chunk.strip()
            # 2. 실제로 문제 번호(예: '1. ', '25. ')로 시작하는 진짜 문제만 담기
            if re.match(r'^\d+\.\s', chunk):
                all_chunks.append(chunk)
all_chunks


['1. 다음을 듣고, 여자가 하는 말의 목적으로 가장 적절한 것을 \n고르시오.\n①학교 종소리 교체 계획을 알리려고\n②학교 수업 시간 단축을 공지하려고\n③등교 시간 변경을 안내하려고\n④학부모 상담 신청서 제출을 독려하려고\n⑤학교 행사 후 교실 정리 정돈을 당부하려고',
 '2. 대화를 듣고, 남자의 의견으로 가장 적절한 것을 고르시오.\n①드라마 캠프는 효율적인 여가 시간 활용 수단이다.\n②좋은 연기를 하려면 다른 사람과의 협력이 중요하다.\n③드라마에는 독특한 개성을 가진 등장인물이 필요하다.\n④원만한 교우 관계를 위해 친구의 말에 귀 기울여야 한다.\n⑤드라마 캠프 참여는 다양한 시각을 갖는 데 도움이 된다.',
 '3. 다음을 듣고, 여자가 하는 말의 요지로 가장 적절한 것을 고르시오.\n①예술가에 관해 알면 작품을 더 잘 이해할 수 있다.\n②지역 사회는 예술가에 대한 지원을 확대해야 한다.\n③예술 작품을 전시할 때 조명 효과를 고려해야 한다.\n④정기적인 미술관 방문은 작품 감상 능력을 높여 준다.\n⑤예술 작품은 보는 사람에 따라 다양한 해석이 가능하다.',
 '4. 대화를 듣고, 그림에서 대화의 내용과 일치하지 않는 것을 고르시오.',
 '5. 대화를 듣고, 남자가 할 일로 가장 적절한 것을 고르시오.\n①트로피 가져오기\n②사진 출력하기\n③이메일 확인하기\n④스티커 주문하기\n⑤게시판 사용 허락받기',
 '6. 대화를 듣고, 여자가 지불할 금액을 고르시오.\n①$100\n②$150\n③$180\n④$200\n⑤$220',
 '7. 대화를 듣고, 남자가 Streamline Broadcasting Workshop에 \n갈 수 없는 이유를 고르시오.\n①동아리 공연에 참여해야 해서\n②교내 방송 준비를 해야 해서\n③야구 경기를 보러 가야 해서\n④생일 파티에 참석해야 해서\n⑤선물을 사러 가야 해서',
 '8. 대화를 듣고, Outstanding Octopuses 행사에 관해 언급되지 \n않은 것을 고르시오.\n①목적\n②프로그램\

In [10]:
import asyncio

batch_size = 30  # 한 번에 요청할 크기

with Connection() as conn:
    with conn.cursor() as cur:
        for i in range(0, len(all_chunks), batch_size):
            batch = all_chunks[i:i + batch_size]
            
            # 429 에러 발생 시 자동으로 대기 후 재시도하는 루프
            while True:
                try:
                    emb_res = await embedder.embed_documents(batch)
                    break  # 성공하면 루프 탈출
                except ModelHTTPError as e:
                    if "429" in str(e):
                        print("⏳ 무료 티어 속도 제한(429) 도달. 25초 대기 후 다시 시도합니다...")
                        await asyncio.sleep(25)
                    else:
                        raise e
            
            # DB 삽입
            for content, emb in zip(batch, emb_res.embeddings):
                cur.execute(
                    """
                    INSERT INTO pdf_documents (content, embedding)
                    VALUES (%s, %s);
                    """,
                    (content, emb)
                )
            
            print(f"저장 진행: {min(i + batch_size, len(all_chunks))}/{len(all_chunks)} 완료")
            
            # 구글 무료 한도를 넘지 않도록 요청 사이에 1.5초 휴식
            await asyncio.sleep(1.5)

    conn.commit()

print("🎉 모든 문제가 에러 없이 안전하게 PostgreSQL에 저장되었습니다!")


저장 진행: 30/180 완료
저장 진행: 60/180 완료
저장 진행: 90/180 완료
⏳ 무료 티어 속도 제한(429) 도달. 25초 대기 후 다시 시도합니다...
저장 진행: 120/180 완료
저장 진행: 150/180 완료
저장 진행: 180/180 완료
🎉 모든 문제가 에러 없이 안전하게 PostgreSQL에 저장되었습니다!
